# Task 3 — The reject contract (9 classes, 4 verdicts)

Nine classes of row and four verdicts: **Load**, **Route to Unknown**,
**Reject**, **Remove as duplicate**.

- **Reject** — the row cannot be loaded at all, and goes to `RejectLog` with a reason.
- **Route to Unknown** — the row loads, a lookup failed, the key is `-1`, and the count is published.
- **Remove as duplicate** — the row is not a second shipment, so taking it out restores the grain.
- **Load** — there is nothing wrong with it.

The four self-check numbers are class counts, not a partition of staging: the
partition is 24 rejected + 60 removed + 1,200 loaded = 1,284. Of the 1,200
loaded rows, 21 carry a `-1` key and 3 are the look-like-faults-but-real
credit notes.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows
  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer


per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)
registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


In [5]:
# The nine classes, counted from the staging masks (rejects) and the routed
# fact rows (Unknown keys). The reject classes are disjoint for March.
n_text = int(wtext.sum())
n_date = int(baddate.sum())
n_wneg = int(wneg.sum())
n_blank = int(blankid.sum())
n_credit = int(stg["FreightCostPKR"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0).sum())
credit_in_loadable = int(loadable["FreightCostPKR"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0).sum())
n_city = int((work["OriginCityKey"] == -1).sum())
n_status = int((work["StatusKey"] == -1).sum())
n_byte = byte_removed
n_trail = removed - byte_removed
classes = pd.DataFrame([
    ("Weight is text ('approx 7000', 'see note', 'TBC', '--', 'unknown', 'n/a', '1.2e4')", "REJECT", n_text),
    ("Date is unparseable (00/00/0000, 2026/13/04, 32/03/2026, 31/02/2026, Mar-2026)", "REJECT", n_date),
    ("Weight is negative", "REJECT", n_wneg),
    ("ShipmentID is blank", "REJECT", n_blank),
    ("FreightCostPKR is negative — a credit note, real business", "LOAD untouched", n_credit),
    ("Origin does not resolve to a city", "ROUTE to Unknown (-1)", n_city),
    ("Status does not resolve to a DimStatus member", "ROUTE to Unknown (-1)", n_status),
    ("Second copy, byte-identical to the first", "REMOVE as duplicate", n_byte),
    ("Second copy, identical except trailing spaces in the ID", "REMOVE as duplicate", n_trail),
], columns=["Class", "Verdict", "Rows"])
print(classes.to_string(index=False))
n_reject = n_text + n_date + n_wneg + n_blank
n_route = n_city + n_status
n_remove = n_byte + n_trail
print(f"\nREJECT {n_reject} / ROUTE {n_route} / REMOVE {n_remove} / LOAD untouched {n_credit}")
print(f"credit notes in loadable: {credit_in_loadable} of {n_credit} (all three survive — none of them fails a reject rule)")

                                                                             Class               Verdict  Rows
Weight is text ('approx 7000', 'see note', 'TBC', '--', 'unknown', 'n/a', '1.2e4')                REJECT     9
    Date is unparseable (00/00/0000, 2026/13/04, 32/03/2026, 31/02/2026, Mar-2026)                REJECT     6
                                                                Weight is negative                REJECT     5
                                                               ShipmentID is blank                REJECT     4
                         FreightCostPKR is negative — a credit note, real business        LOAD untouched     3
                                                 Origin does not resolve to a city ROUTE to Unknown (-1)    14
                                     Status does not resolve to a DimStatus member ROUTE to Unknown (-1)     7
                                          Second copy, byte-identical to the first   REMOVE as duplicate    48
 

### Part B — the `RejectLog` schema (6 columns)

The test: somebody who has never seen this pipeline can take a row out of
the table, fix the source, and put it through again. Six columns is a good
answer; four is thin; nine means storing the same thing twice.

| Column | What it is for |
|---|---|
| `RejectID` | Stable identity for the rejected row (`R-001`, …) — the handle you quote when you ask the source team to fix it |
| `SourceFile` | Which of the three files it came from — the team that owns the fix |
| `SourceLine` | The exact line/row it came from (CSV line with header = 1; rail Excel row) — find it without searching |
| `ShipmentID_Raw` | The key exactly as it arrived, blank included — the row's identity before any transform |
| `RejectReason` | The class and the rule violated (e.g. `weight_not_numeric`) — why it cannot load |
| `SourceRow` | The full original row, pipe-separated, as it arrived — everything needed to re-run it after the fix |

A reject table that only says "row 418 failed" is a log, not an audit table.

In [6]:
assert (n_text, n_date, n_wneg, n_blank) == (9, 6, 5, 4), "reject classes must be 9 / 6 / 5 / 4"
assert n_reject == 24 and n_route == 21 and n_remove == 60 and n_credit == 3, "self-check: 24 / 21 / 60 / 3"
assert credit_in_loadable == 3, "all three credit notes must survive into the loadable set"
assert n_reject + n_remove + len(fact) == len(stg) == 1284, "partition: 24 + 60 + 1200 = 1284"
print("SELF-CHECK PASS — 24 rejected / 21 routed to Unknown / 60 removed as duplicates / 3 loaded untouched")

# Analysis
# Reject vs route — reject means the row cannot be a fact row at all
# (missing key, unparseable date, unmeasurable weight): loading it would
# corrupt totals or the grain. Route means the row IS a shipment with valid
# measures and only a REFERENCE lookup failed: it loads with -1 so totals
# stay true and the gap stays visible and counted. Costs of swapping:
# rejecting a routable row deletes real freight (the 14 city rows carry
# 195,150 kg — the dashboard goes green and the quarter loses real shipments
# with no trace); routing a rejectable row loads a row with no date, key or
# weight, breaking the grain and every time/total analysis built on it.

SELF-CHECK PASS — 24 rejected / 21 routed to Unknown / 60 removed as duplicates / 3 loaded untouched
